# Analyse de l'Incertitude et Calibration (SAN-2)
**Auteur :** Fatima-Zahra Massane  
**Branche :** `fatimazahra/uncertainty-analysis`  
**Données :** FedISIC2019 - Split `val` (développement uniquement)

## Objectifs
1. Comparer la distribution d'incertitude (Entropie) entre Single, MC-Dropout et Ensemble.
2. Évaluer la calibration via le Diagramme de Fiabilité (ECE).
3. Analyser le compromis Confiance vs Accuracy (Prédiction Sélective).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Style professionnel pour l'article
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 300

# Chemins (adaptés à la structure de Salma)
ROOT = Path("..").resolve()
OUTPUT_DIR = ROOT / "outputs"
LOGITS_DIR = OUTPUT_DIR / "logits"
MCD_DIR = OUTPUT_DIR / "mcdropout"
ENS_DIR = OUTPUT_DIR / "ensembles"
FIG_DIR = OUTPUT_DIR / "figures" / "uncertainty"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f" Configuration chargée. Figures sauvegardées dans : {FIG_DIR}")

In [ ]:
def load_val_data():
    """Charge les prédictions sur val (SEUL split autorisé)."""
    data = {}
    
    # 1. Single (Baseline) : moyenne des 3 seeds de logits -> softmax
    single_probs = []
    for seed in [0, 1, 2]:
        path = LOGITS_DIR / f"val_seed{seed}.npy"
        if path.exists():
            logits = np.load(path).astype(np.float64)
            # Softmax stable
            shifted = logits - logits.max(axis=1, keepdims=True)
            probs = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)
            single_probs.append(probs)
    if single_probs:
        data['single'] = np.mean(single_probs, axis=0)
    
    # 2. MC-Dropout : moyenne des 3 seeds de probas
    mcd_probs = []
    for seed in [0, 1, 2]:
        path = MCD_DIR / f"mcdropout_val_seed{seed}.npy"
        if path.exists():
            mcd_probs.append(np.load(path).astype(np.float64))
    if mcd_probs:
        data['mc_dropout'] = np.mean(mcd_probs, axis=0)
    else:
        print(" MC-Dropout val non trouvé (lance src/mc_dropout.py --val_only)")
        
    # 3. Ensemble
    ens_path = ENS_DIR / "ensemble_val.npy"
    if ens_path.exists():
        data['ensemble'] = np.load(ens_path).astype(np.float64)
    else:
        print(" Ensemble val non trouvé (lance src/ensembles.py --val_only)")
        
    # Labels
    labels_path = LOGITS_DIR / "val_labels.npy"
    if labels_path.exists():
        data['labels'] = np.load(labels_path).astype(np.int64)
        
    return data

val_data = load_val_data()
print(f" Méthodes chargées : {[k for k in val_data.keys() if k != 'labels']}")

In [ ]:
def compute_entropy(probs):
    probs = np.clip(probs, 1e-10, 1.0)
    return -np.sum(probs * np.log(probs), axis=1)

fig, ax = plt.subplots(figsize=(10, 6))
colors = {'single': '#2196F3', 'mc_dropout': '#FF9800', 'ensemble': '#4CAF50'}

for method, probs in val_data.items():
    if method == 'labels': continue
    entropy = compute_entropy(probs)
    sns.kdeplot(entropy, ax=ax, label=method.replace('_', ' ').title(), 
                fill=True, alpha=0.4, color=colors.get(method, 'gray'), linewidth=2)

ax.set_xlabel("Entropie de la prédiction (nats)", fontsize=13)
ax.set_ylabel("Densité", fontsize=13)
ax.set_title("Distribution de l'incertitude sur val\n(MC-Dropout = 20 passes)", fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig1_entropy_distribution_val.png", bbox_inches='tight')
print(" Figure 1 sauvegardée : fig1_entropy_distribution_val.png")
plt.show()

In [ ]:
def compute_ece(probs, labels, n_bins=15):
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    accuracies = (predictions == labels).astype(float)
    
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_infos = []
    
    for i in range(n_bins):
        mask = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i+1])
        if mask.sum() == 0:
            bin_infos.append((0, 0, 0))
            continue
        bin_acc = accuracies[mask].mean()
        bin_conf = confidences[mask].mean()
        bin_size = mask.sum()
        ece += (bin_size / len(labels)) * abs(bin_acc - bin_conf)
        bin_infos.append((bin_conf, bin_acc, bin_size))
    return ece, bin_infos

fig, axes = plt.subplots(1, len([k for k in val_data.keys() if k != 'labels']), 
                         figsize=(15, 5), sharey=True)
if len(axes) == 1: axes = [axes]

for ax, (method, probs) in zip(axes, [(k, v) for k, v in val_data.items() if k != 'labels']):
    ece, bins = compute_ece(probs, val_data['labels'])
    confidences = [b[0] for b in bins]
    accuracies = [b[1] for b in bins]
    counts = [b[2] for b in bins]
    
    ax.bar(confidences, counts, width=1/15, alpha=0.3, color='gray', edgecolor='black')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Parfaite')
    ax.plot(confidences, accuracies, 'o-', linewidth=2, color='#E91E63', label=f'ECE={ece:.4f}')
    
    ax.set_title(method.replace('_', ' ').title(), fontsize=13, fontweight='bold')
    ax.set_xlim(0, 1)
    ax.legend(fontsize=9)

axes[0].set_ylabel("Accuracy / Effectif", fontsize=12)
fig.suptitle("Diagrammes de fiabilité sur val (15 bins)", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig2_reliability_diagram_val.png", bbox_inches='tight')
print(" Figure 2 sauvegardée : fig2_reliability_diagram_val.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
thresholds = np.linspace(0.5, 1.0, 50)

for method, probs in val_data.items():
    if method == 'labels': continue
    preds = probs.argmax(axis=1)
    confs = probs.max(axis=1)
    
    accs, coverage = [], []
    for t in thresholds:
        mask = confs >= t
        if mask.sum() == 0:
            accs.append(np.nan); coverage.append(0)
        else:
            accs.append((preds[mask] == val_data['labels'][mask]).mean())
            coverage.append(mask.mean())
    
    ax.plot(thresholds, accs, 'o-', linewidth=2, markersize=4, 
            label=f"{method.replace('_', ' ').title()} (accuracy)", 
            color=colors.get(method, 'gray'))

ax.axhline(y=0.77, color='gray', linestyle=':', linewidth=1, label='Accuracy val (~0.77)')
ax.set_xlabel("Seuil de confiance minimum", fontsize=13)
ax.set_ylabel("Accuracy sélective", fontsize=13)
ax.set_title("Accuracy vs Seuil de confiance (val)\nPlus le seuil est haut, moins de cas mais meilleure accuracy", 
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.set_xlim(0.5, 1.0)
ax.set_ylim(0.5, 1.05)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig3_confidence_vs_accuracy_val.png", bbox_inches='tight')
print(" Figure 3 sauvegardée : fig3_confidence_vs_accuracy_val.png")
plt.show()